# Quick DHI Detectability Check

Since the last dataset, that is more physically realistic and then therefore more complicated, didn;t work well for the simple model I want to wuickly see if a simple model can tell an injected DHI apart from real background at all? Before investing more time in the full physically rigorous pipeline (`synthetic_dhi_generation.ipynb`).

Deliberately minimal on purpose:
- No horizon-conformance (flat injection only, no structural-high matching)
- No hard-negative (just clean background as negatives)
- No spatial train/test split (this is a sanity check, not a generalization test)
- Reuses real background traces already sitting in the existing local dataset (outside each patch's injection footprint mask) instead of re-reading the raw SEG-Y - keeps this fast and avoids depending on the OneDrive-hosted SEG-Y being fully synced. (This is just a temporary solution as I am waiting for my onedrive to download files locally)

In [ ]:
import glob
import sys
sys.path.append('..')

import numpy as np
import pandas as pd

from src.dhi_pipeline.injection import inject_dhi_anomaly, RC_GAS_SAND, estimate_amplitude_scale

LOCAL_DATASET_DIR = '../data/dataset'

## 1. Pool real background traces

Each existing patch's `mask` marks where its injection sits - so `amplitude[~mask]` is real F3 background that was never touched by any injection. Pooling this across many existing patches gives real-data variety without needing to touch the SEG-Y at all.

In [2]:
patch_files = sorted(glob.glob(LOCAL_DATASET_DIR + '/patches/*.npz'))
print(f'{len(patch_files)} local patch files available')

pool = []
for pf in patch_files:
    d = np.load(pf)
    amp = d['attribute_stack'][0]
    mask = d['mask']
    if amp.shape[-1] != 125:   # skip a few patches clipped near the survey's time boundary
        continue
    pool.append(amp[~mask])

background_pool = np.concatenate(pool, axis=0)
print('total background trace pool:', background_pool.shape)

53 local patch files available


total background trace pool: (466879, 125)


## 2. Inject simple flat doublets across severities

Reuses `inject_dhi_anomaly` (the flat, non-horizon-following injector already in the pipeline) at a spread of severities from subtle to obvious - same idea as `SEVERITY_TIERS` in the main pipeline, just simplified to 4 fixed points instead of continuous tiers, and no flat-spot/polarity-reversal extras.

In [3]:
dt_ms = 4.0
n_samples = background_pool.shape[1]
time_axis_ms = np.arange(1, n_samples + 1) * dt_ms
window_size = 30
xl_axis = np.arange(window_size)
amp_scale = estimate_amplitude_scale(background_pool, reference_rc=0.05)

SEVERITIES = {
    'subtle':   dict(thickness_m=2.0,  rc_frac=0.30),
    'mild':     dict(thickness_m=4.0,  rc_frac=0.50),
    'moderate': dict(thickness_m=7.0,  rc_frac=0.70),
    'obvious':  dict(thickness_m=14.0, rc_frac=1.00),
}
N_PER_SEVERITY = 10
N_NEGATIVES = 40

rng = np.random.default_rng(0)
all_idx = rng.permutation(len(background_pool))
cursor = 0

def next_window():
    global cursor
    idx = all_idx[cursor: cursor + window_size]
    cursor += window_size
    return background_pool[idx].copy()

def features(window):
    return [window.mean(), window.std(), np.abs(window).max()]

rows, feats = [], []
for sev_name, sev in SEVERITIES.items():
    for _ in range(N_PER_SEVERITY):
        window = next_window()
        injected, _ = inject_dhi_anomaly(
            window, time_axis_ms, xl_axis, top_time_ms=250,
            thickness_m=sev['thickness_m'], velocity_mps=2180,
            reflection_coefficient=RC_GAS_SAND * sev['rc_frac'],
            freq_hz=60, amplitude_scale=amp_scale,
        )
        feats.append(features(injected))
        rows.append(dict(is_dhi=True, severity=sev_name))

for _ in range(N_NEGATIVES):
    window = next_window()
    feats.append(features(window))
    rows.append(dict(is_dhi=False, severity='background'))

labels = pd.DataFrame(rows)
X = np.array(feats)
y = labels['is_dhi'].values
print(labels['is_dhi'].value_counts())
print('X shape:', X.shape)

is_dhi
True     40
False    40
Name: count, dtype: int64
X shape: (80, 3)
